# 02 — Nettoyage & Harmonisation des Données
**Applique toutes les décisions verrouillées (voir README.md)**

Règles :
- IMC : binning OMS ordinal (PAS de midpoint)
- Tension : binning ordinal 3 niveaux
- HTA + pré-éclampsie : variable combinée
- "Non_renseigne" (Négatifs) → NaN
- Parsing textes Positifs (âge, SA, parité, glycémie)
- flag_source_batch créé mais PAS utilisé comme feature


In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
from data_cleaning import (
    bin_imc_continu, parse_imc_categoriel_positifs,
    bin_tension_continue, parse_tension_categorielle_positifs,
    combine_hta_preeclampsie, normalize_oui_non,
    parse_age_positifs, parse_sa_positifs, parse_parite_positifs,
    parse_glycemie_virgule, non_renseigne_to_nan, create_flag_source_batch
)

df_france = pd.read_csv('../data/raw/dataset_dg_france_30000_final.csv')
df_positifs = pd.read_csv('../data/raw/Cas_positifs_243.csv')
df_negatifs = pd.read_csv('../data/raw/cas_negatifs_212.csv')
print("Chargement OK:", df_france.shape, df_positifs.shape, df_negatifs.shape)


Chargement OK: (30000, 30) (243, 36) (212, 19)


## 2.1 Nettoyage — France (dataset d'entraînement)

In [2]:
fr = df_france.copy()

# Binning IMC ordinal
fr['imc_ordinal'] = bin_imc_continu(fr['imc'])

# Binning tension ordinal
fr['tension_ordinal'] = bin_tension_continue(fr['ta_systolique'], fr['ta_diastolique'])

# Variable combinée HTA/pré-éclampsie
fr['hta_ou_preeclampsie'] = combine_hta_preeclampsie(fr['hta_chronique'], fr['atcd_preeclampsie'])

# Normalisation Oui/Non sur toutes les colonnes binaires
binary_cols_fr = ['atcd_familial_diabete_1er_deg','atcd_gdm','atcd_macrosomie',
                   'sopk','sedentarite','tabagisme','gdm_label']
for c in binary_cols_fr:
    fr[c] = normalize_oui_non(fr[c])

fr['flag_source_batch'] = 0  # N/A pour France, mis à 0 par convention

print("France nettoyée:", fr.shape)
fr[['imc','imc_ordinal','tension_ordinal','hta_ou_preeclampsie']].head()


France nettoyée: (30000, 34)


,imc,imc_ordinal,tension_ordinal,hta_ou_preeclampsie
0,22.011893,2,1.0,Non
1,25.151988,3,2.0,Non
2,29.276214,3,1.0,Non
3,27.857894,3,1.0,Non
4,26.533349,3,1.0,Oui


## 2.2 Nettoyage — Cameroun Positifs (243)

In [3]:
pos = df_positifs.copy()

# Renommage colonnes vers noms harmonisés (France/Négatifs)
pos_rename = {
    'Age (en années)': 'age_maternel_raw',
    "Nombre d’accouchements (parité)": 'parite_raw',
    'ATCD familiaux de diabète (1er degré)': 'atcd_familial_diabete_1er_deg',
    "Niveau d'instruction ": 'niveau_etude',
    'IMC( kg/m²)': 'imc_raw',
    'Pression arterielle(mm Hg)': 'tension_raw',
    "Sédentaire?(moins de 150 minutes d'activité physique par semaine).": 'sedentarite',
    'Tabagisme(actif/passif)': 'tabagisme',
    'ATCD de diabète gestationnel': 'atcd_gdm',
    'ATCD de macrosomie (>4kg)': 'atcd_macrosomie',
    'Antécédent de SOPK (kystes ovariens)': 'sopk',
    'ATCD d’HTA(Hypertension artérielle) ou pré-éclampsie': 'hta_preeclampsie_raw',
    "Âge gestationnel (en SA) à l’inclusion": 'sa_premiere_consult_raw',
    'Diagnostic DG': 'gdm_label',
    'Glycémie à jeun (si dispo)': 'glycemie_jeun_raw',
}
pos = pos.rename(columns=pos_rename)

# Parsing des colonnes texte
pos['age_maternel'] = parse_age_positifs(pos['age_maternel_raw'])
pos['sa_premiere_consult'] = parse_sa_positifs(pos['sa_premiere_consult_raw'])
pos['parite'] = parse_parite_positifs(pos['parite_raw'])
pos['imc_ordinal'] = parse_imc_categoriel_positifs(pos['imc_raw'])
pos['tension_ordinal'] = parse_tension_categorielle_positifs(pos['tension_raw'])
pos['glycemie_jeun_1T'] = parse_glycemie_virgule(pos['glycemie_jeun_raw'])

# Variable combinée HTA -> ici directement disponible (déjà combinée dans le formulaire terrain)
pos['hta_ou_preeclampsie'] = normalize_oui_non(pos['hta_preeclampsie_raw'])

# Normalisation Oui/Non (gère les 'oui' minuscule détectés dans atcd_gdm)
for c in ['atcd_familial_diabete_1er_deg','atcd_gdm','atcd_macrosomie','sopk','sedentarite','tabagisme','gdm_label']:
    pos[c] = normalize_oui_non(pos[c])

pos['flag_source_batch'] = 0  # N/A pour Positifs (pas de pattern détecté ici)

print("Positifs nettoyés:", pos.shape)
print(f"\nAnomalies parité (devenues NaN, ex: 'A'): {pos['parite'].isna().sum() - pos['parite_raw'].isna().sum()}")
pos[['age_maternel','imc_ordinal','tension_ordinal','parite','hta_ou_preeclampsie']].head()


Positifs nettoyés: (243, 44)

Anomalies parité (devenues NaN, ex: 'A'): 1


,age_maternel,imc_ordinal,tension_ordinal,parite,hta_ou_preeclampsie
0,37.0,4,2,Multipare_2,Non
1,35.0,3,1,Multipare_3,Non
2,22.0,3,1,Primipare,Non
3,26.0,2,1,Nullipare,Non
4,35.0,3,1,Nullipare,Non


## 2.3 Nettoyage — Cameroun Négatifs (212)

In [4]:
neg = df_negatifs.copy()

# "Non_renseigne" -> NaN sur toutes les colonnes concernées
cols_with_nr = ['parite','zone_residence','hta_chronique','sedentarite','tabagisme',
                'alcoolisme','atcd_preeclampsie','sopk','grossesse_multiple']
neg = non_renseigne_to_nan(neg, cols_with_nr)

# Binning IMC et tension (déjà continus)
neg['imc_ordinal'] = bin_imc_continu(neg['imc'])
neg['tension_ordinal'] = bin_tension_continue(neg['ta_systolique'], neg['ta_diastolique'])

# Variable combinée HTA/pré-éclampsie
neg['hta_ou_preeclampsie'] = combine_hta_preeclampsie(neg['hta_chronique'], neg['atcd_preeclampsie'])

# Normalisation
for c in ['atcd_familial_diabete_1er_deg','atcd_gdm','atcd_macrosomie','sopk','sedentarite','tabagisme','gdm_label']:
    neg[c] = normalize_oui_non(neg[c])

# 🚨 FLAG SOURCE BATCH — signature des 108 lignes à 2 sous-cohortes
check_cols = ['sopk','sedentarite','tabagisme','hta_chronique','atcd_preeclampsie']
neg['flag_source_batch'] = create_flag_source_batch(neg, check_cols)

print("Négatifs nettoyés:", neg.shape)
print(f"\nflag_source_batch = 1 (sous-cohorte incomplète) : {neg['flag_source_batch'].sum()} femmes")
neg[['imc_ordinal','tension_ordinal','hta_ou_preeclampsie','flag_source_batch']].head()


Négatifs nettoyés: (212, 23)

flag_source_batch = 1 (sous-cohorte incomplète) : 108 femmes


c:\PROJECTS\GDM_VALIDATION_PROJECT\notebooks\../src\data_cleaning.py:206: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace('Non_renseigne', np.nan)


,imc_ordinal,tension_ordinal,hta_ou_preeclampsie,flag_source_batch
0,4,1,NaN,1
1,3,1,NaN,1
2,3,1,NaN,1
3,3,1,NaN,1
4,5,1,NaN,1


## 2.4 Combinaison Positifs + Négatifs = Dataset de validation externe Cameroun (455 cas)

In [5]:
common_cols = ['age_maternel','imc_ordinal','tension_ordinal','sa_premiere_consult',
               'parite','atcd_familial_diabete_1er_deg','atcd_gdm','atcd_macrosomie',
               'sopk','sedentarite','tabagisme','hta_ou_preeclampsie','niveau_etude',
               'gdm_label','flag_source_batch']

# niveau_etude absent des Négatifs actuels -> vérifier
if 'niveau_etude' not in neg.columns:
    neg['niveau_etude'] = np.nan
    print("⚠️ niveau_etude absent de Négatifs -> rempli en NaN (à confirmer avec David)")

cameroun_combined = pd.concat([
    pos[common_cols].assign(source='positifs'),
    neg[common_cols].assign(source='negatifs')
], ignore_index=True)

print(f"Dataset validation Cameroun combiné: {cameroun_combined.shape}")
print(cameroun_combined['gdm_label'].value_counts())


Dataset validation Cameroun combiné: (455, 16)
gdm_label
Oui    243
Non    212
Name: count, dtype: int64


## 2.5 Sauvegarde des datasets nettoyés

In [6]:
fr.to_csv('../data/processed/france_train_clean.csv', index=False)
pos.to_csv('../data/processed/cameroun_positifs_clean.csv', index=False)
neg.to_csv('../data/processed/cameroun_negatifs_clean.csv', index=False)
cameroun_combined.to_csv('../data/processed/cameroun_combined_validation.csv', index=False)

print("✅ 4 fichiers sauvegardés dans data/processed/")
print("   - france_train_clean.csv")
print("   - cameroun_positifs_clean.csv")
print("   - cameroun_negatifs_clean.csv")
print("   - cameroun_combined_validation.csv (455 cas — POUR VALIDATION UNIQUEMENT)")


✅ 4 fichiers sauvegardés dans data/processed/
   - france_train_clean.csv
   - cameroun_positifs_clean.csv
   - cameroun_negatifs_clean.csv
   - cameroun_combined_validation.csv (455 cas — POUR VALIDATION UNIQUEMENT)


## 2.6 Checklist de validation du nettoyage

À vérifier avant de passer au notebook 03 :
- [ ] `imc_ordinal` : valeurs 1-5 uniquement, pas de valeurs hors plage
- [ ] `tension_ordinal` : valeurs 1-3 uniquement
- [ ] Aucune valeur 'oui'/'non' minuscule résiduelle
- [ ] `flag_source_batch` = 108 sur Négatifs (pas plus, pas moins)
- [ ] Anomalie parité 'A' bien convertie en NaN (pas en valeur aberrante)

➡️ **Suite : notebook 03_feature_engineering.ipynb**
